[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/04-recall-tuning/02-weighting_fields_for_better_precision.ipynb)

# Weighting Fields for Better Precision

`overall_score` is not a single measurement. It is a combination of every field's individual score, blended together into one number. By default, M|BOX blends fields without any special bias toward one over another. But real records rarely have every field carry equal importance. A product's own name is usually a stronger signal of identity than a line of free-text description. A legal ID is usually a stronger signal than a nickname.

`weight` lets you tell the engine which fields should count for more when it calculates `overall_score`. `minimum_quality` lets you go further and set a hard floor a field must clear, regardless of how well everything else scores. Get the balance wrong, and two genuinely different records can end up ranked in the wrong order. Get it right, and an ambiguous query resolves the way a person reviewing the data by hand actually would.

In this notebook you will:

1. See a genuinely ambiguous query, where two different products both have a real claim to being the correct match
2. Watch the ranking change depending on which field is weighted more heavily
3. Learn what `TableRecallConfig` and `TableRecallFieldConfig` are, and how they relate to the inline shortcuts you've used so far
4. Use `minimum_quality` to disqualify a field outright, rather than just discounting it
5. Avoid a common surprise when moving from inline shortcuts to a full recall config
6. Walk away with practical guidelines for setting weights you can defend

In [ ]:
# !pip install mbox

## 1. A query with two legitimate candidates

Building a good example of "weighting matters" requires two records that are each a plausible match for the same query, but for different reasons. Here is a small product catalog built for exactly that.

`"B88-EXT"` is literally named `"Extended Battery Pack"`. Its description, though, only loosely touches on the same idea. `"X77-BAT"` has a very different name, but its description contains almost the exact phrase we are about to search for. Both products have *some* relevant signal in *both* fields, which matters, and we will come back to why in a moment.

In [1]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "product_id": ["B88-EXT", "X77-BAT", "C99-SNS", "D45-REL"],
    "product_name": [
        "Extended Battery Pack",
        "Extended Cell Battery",
        "Motion Sensor Camera",
        "Smart Relay Switch"
    ],
    "description": [
        "Compact accessory offering extended battery life for outdoor gear",
        "Extended battery pack accessory for compact cameras",
        "Motion sensor camera with night vision for home security",
        "Smart relay switch for home automation systems"
    ]
})

index = TableIndexer.create_index(
    df=df,
    index_columns=["product_name", "description"],
    tmp_dir="tmp_index"
)

df

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,product_name,description
0,B88-EXT,Extended Battery Pack,Compact accessory offering extended battery li...
1,X77-BAT,Extended Cell Battery,Extended battery pack accessory for compact ca...
2,C99-SNS,Motion Sensor Camera,Motion sensor camera with night vision for hom...
3,D45-REL,Smart Relay Switch,Smart relay switch for home automation systems


### A quick but important note before you search

When you query more than one field in a single `match()` call, M|BOX is not scoring each field in isolation and combining the scores afterward as if they were independent lookups. It is looking for rows that are genuinely relevant candidates *across all the fields you queried at once*. If a row has strong signal in one field but effectively none in another, it may never enter the candidate set at all, and it will not show up in your results no matter how you weight the fields afterward.

This is exactly why both `product_name` and `description` above contain at least some overlap with the query for both `"B88-EXT"` and `"X77-BAT"`. Neither field is a hard zero for either row. That is what allows both rows to genuinely compete, so that weighting can decide the outcome, rather than one row being excluded from consideration before weighting ever comes into play.

## 2. A baseline query, no weighting applied

Let's search for `"Extended Battery Pack"` across both fields at once, with no weighting.

In [2]:
baseline_results = index.match(
    product_name="Extended Battery Pack",
    description="Extended Battery Pack",
    include_field_scores=True,
    min_total_match_value=0,
    max_results=4
)

baseline_results

,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,B88-EXT,64,100,29
1,0,1,Extended Cell Battery,Extended battery pack accessory for compact ca...,X77-BAT,64,42,86


Both `"B88-EXT"` and `"X77-BAT"` should appear in the results now. `"B88-EXT"` should score well on `product_name_score`, since that is its actual name. `"X77-BAT"` should score well on `description_score`, since its description contains nearly the same phrase. With no weighting, whichever one ranks higher is deciding a real question, is a product's own name more trustworthy than a phrase inside its description, by default, rather than by a deliberate choice you made.

## 3. Weighting toward `product_name`

Suppose you believe a product's own name should be the dominant signal, and a description is just supporting context. `weights` lets you say so directly.

In [3]:
name_heavy_results = index.match(
    product_name="Extended Battery Pack",
    description="Extended Battery Pack",
    weights={"product_name": 90, "description": 10},
    include_field_scores=True,
    min_total_match_value=0,
    max_results=4
)

name_heavy_results

,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,B88-EXT,92,100,29
1,0,1,Extended Cell Battery,Extended battery pack accessory for compact ca...,X77-BAT,46,42,86


`"B88-EXT"` should now be ranked clearly ahead of `"X77-BAT"`. Its strong `product_name_score` is now carrying 90 percent of the weight in `overall_score`, so `"X77-BAT"`'s strong description match cannot compete, no matter how good the phrase overlap is there.

## 4. Weighting toward `description`

Now flip it around. Suppose your actual use case is matching customer support messages against product descriptions. Product names barely come up in that context, so description is the field that should carry the query.

In [4]:
description_heavy_results = index.match(
    product_name="Extended Battery Pack",
    description="Extended Battery Pack",
    weights={"product_name": 10, "description": 90},
    include_field_scores=True,
    min_total_match_value=0,
    max_results=4
)

description_heavy_results

,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score,description_score
0,0,1,Extended Cell Battery,Extended battery pack accessory for compact ca...,X77-BAT,81,42,86
1,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,B88-EXT,36,100,29


The ranking should flip. `"X77-BAT"` should now come out ahead, because its description match is now doing 90 percent of the work, and `"B88-EXT"`'s strong name match is worth comparatively little.

Same query. Same indexed data. Two different, both defensible, rankings, purely because the weights encode a different judgment about which field to trust. This is the entire point of `weight`. It is less a technical tuning knob and more a place to write down a business decision explicitly, instead of leaving it to whatever the default blend happens to produce.

## 5. Meet `TableRecallConfig` and `TableRecallFieldConfig`

Everything so far used inline keyword arguments directly on `match()`, `weights={}` and `modes={}`. That is convenient for quick exploration, but it has two limits. First, there is no inline shortcut for everything you might want to control, `minimum_quality`, covered next, has no inline equivalent at all. Second, inline keyword arguments live only in the cell that calls `match()`, there is no way to save them, hand them to a teammate, or reuse them in a different part of a pipeline.

`TableRecallFieldConfig` and `TableRecallConfig` solve both problems, by making a full recall configuration into an explicit, reusable object, the same way `TableFieldConfig` and `TableConfig` made an index schema into an explicit, reusable object in the previous directory.

**`TableRecallFieldConfig`** describes how one field should be evaluated during a search. It bundles together everything you have set inline so far, plus one thing you have not seen yet:

| Attribute | What it does | Inline equivalent |
|---|---|---|
| `input_column` | The column name in your query | (the keyword argument name itself) |
| `indexed_column` | The column in the index to compare against | (the keyword argument name itself) |
| `weight` | How much this field counts toward `overall_score` | `weights={"field": value}` |
| `mode` | Which comparison algorithm to use | `modes={"field": TableRecallMode.X}` |
| `minimum_quality` | A hard score floor this field must clear | no inline equivalent |

**`TableRecallConfig`** is the container that holds a list of `TableRecallFieldConfig` objects, one per field, plus the global settings you have already been passing directly to `match()`, things like `max_results` and `min_total_match_value`. Instead of scattering those across keyword arguments, they all live together in one object you can build once and reuse.

Let's see this in action by building the field-level hard floor that inline keywords cannot express.

## 6. `minimum_quality`: a hard floor, not a discount

Weighting changes how much a field's score counts toward the total, a low-weighted field can still contribute a little. `minimum_quality` is different in kind, not just in degree: it disqualifies a candidate outright if a specific field does not clear a bar, no matter how high its `overall_score` would otherwise be.

This matters when a weak match on one field should never be acceptable, regardless of how strong everything else is. Suppose your business rule is simple: never surface a product whose name does not at least strongly resemble the query, even if the description is a great match.

In [5]:
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

strict_name_floor = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(
            input_column="product_name",
            indexed_column="product_name",
            minimum_quality=85,   # product_name must score at least 85, or the candidate is dropped
            weight=30,
            mode=TableRecallMode.APPROX
        ),
        TableRecallFieldConfig(
            input_column="description",
            indexed_column="description",
            minimum_quality=0,
            weight=70,
            mode=TableRecallMode.APPROX
        )
    ],
    max_results=4,
    min_total_match_value=0,
    non_search_output_fields=["product_id"],
    include_field_scores=True
)

gated_results = index.match(
    queries=pd.DataFrame({
        "product_name": ["Extended Battery Pack"],
        "description": ["Extended Battery Pack"]
    }),
    config=strict_name_floor
)

gated_results

,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,B88-EXT,50,100,29


Compare this against Step 4's description-heavy result. Even though `description` still carries most of the weight here, 70 versus 30, `"X77-BAT"`'s `product_name`, `"Extended Cell Battery"`, is missing the word `"Pack"` and reorders the remaining words compared to the query `"Extended Battery Pack"`. If that is enough of a difference to keep its `product_name_score` under the `minimum_quality=85` floor we set, `"X77-BAT"` should be filtered out of the results entirely here, not just ranked lower the way it would be under pure weighting.

That is the practical difference between weighting and gating. Weighting adjusts how much influence a field has. `minimum_quality` enforces a rule that cannot be overridden by how well the rest of the record scores.

## 7. A common surprise: missing payload columns

Look closely at the `non_search_output_fields=["product_id"]` line in the code above. This is easy to miss, and easy to be confused by if you do miss it.

When you use the simple inline shortcut, `index.match(product_name=..., description=...)`, every column that is part of your index but not being searched, your payload columns, is automatically included in the results. You saw this throughout earlier notebooks: `product_id_candidate` just showed up, without you asking for it directly.

That automatic behavior does not carry over to `TableRecallConfig`. Once you move to the full config object, only the payload columns you explicitly list in `non_search_output_fields` will appear in your results. Leave it out, and columns like `product_id` will silently disappear from your output, not because anything failed, but because the full config object requires you to say what you want, rather than assuming everything.

If you ever switch a query from the inline shortcut to a `TableRecallConfig` and a column you were relying on vanishes from the result, this is almost always why.

## 8. Inline shortcuts versus the full config, side by side

Now that you have seen both approaches, here is the complete picture of how they relate:

| Inline keyword on `match()` | `TableRecallConfig` / `TableRecallFieldConfig` equivalent |
|---|---|
| `weights={"field": 80}` | `TableRecallFieldConfig(weight=80, ...)` |
| `modes={"field": TableRecallMode.APPROX}` | `TableRecallFieldConfig(mode=TableRecallMode.APPROX, ...)` |
| `max_results=5` | `TableRecallConfig(max_results=5, ...)` |
| `min_total_match_value=50` | `TableRecallConfig(min_total_match_value=50, ...)` |
| *(payload columns included automatically)* | `TableRecallConfig(non_search_output_fields=[...], ...)` |
| *(no inline equivalent)* | `TableRecallFieldConfig(minimum_quality=60, ...)` |

Use inline keywords for quick, one-off exploration in a notebook. Reach for a full `TableRecallConfig` once you need `minimum_quality`, once you want to reuse the same tuned configuration in more than one place, or once you are ready to save it to disk, which is exactly what the last notebook in this directory covers.

## 9. Best practices for setting weights

**Start with default weighting, and only tune once you see a real problem.** It is tempting to weight every field the moment you learn how, but weights encode a judgment call, and judgment calls are easiest to get wrong when you are guessing rather than reacting to an actual bad ranking you observed.

**Use `weight` for "more important," and `minimum_quality` for "non-negotiable."** If a low score on a field should just count for less, weight it down. If a low score on a field should disqualify the candidate outright, no matter what else looks good, use `minimum_quality` instead. Conflating the two usually means whichever field has the highest weight quietly overrides a rule you meant to be absolute.

**Weight ratios matter more than absolute values.** A weight of 80 against a weight of 20 produces the same balance as 8 against 2. What matters is the relative proportion between fields, not the raw numbers. Pick a scale you and your team find readable, out of 100 is the most common convention, and stay consistent with it.

**Test weight changes against known-correct matches before deploying them.** A weight change that fixes one ambiguous query can easily flip the ranking on a dozen others you were not looking at. Keep a small, hand-verified set of query and expected-match pairs, and re-run it every time you adjust weights, the same discipline you would apply to a regression test suite.

**Document why a weight is what it is.** A weight of 90 on `product_name` means nothing to the next person who touches this code unless they know it is because product names are considered a stronger identity signal than descriptions in your domain. Treat a `TableRecallConfig` the same way you would treat any other piece of business logic, with comments or a short doc explaining the reasoning, not just the numbers.

**Prefer a few fields with clear, deliberate weights over many fields with vague ones.** Every additional weighted field is another variable that can interact with the others in ways that are hard to predict. If a field's signal is marginal, it is often clearer to leave it out of the weighting scheme entirely, or give it a small, stable weight, than to tune it precisely alongside everything else.

**Remember that multi-field queries require overlap in every field you search.** As you saw in Step 1, a candidate needs at least some relevant signal in every field you are querying to be considered at all. Weighting only decides the outcome among candidates that were already in the running, it cannot rescue a candidate that never made it into the results to begin with.

## Next steps

- **`03-numeric_matching_ranges_and_thresholds.ipynb`** - the same weighting concepts, applied to numeric fields like price and quantity
- **`04-reusable_recall_configs_as_json.ipynb`** - save a tuned `TableRecallConfig`, weights, quality floors, and all, and reuse it across a pipeline

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*